# 🐍 SQLite con Python
Este cuaderno explica el uso de **SQLite** en Python, cubriendo la creación de bases de datos, tablas y operaciones CRUD (Create, Read, Update, Delete). Además, veremos cómo realizar consultas con `SELECT` y extraer datos con `fetchone()`, `fetchmany()` y `fetchall()`.

## 1. Introducción a SQLite
SQLite es una base de datos **ligera y embebida** que no requiere un servidor independiente. En Python se trabaja con el módulo integrado `sqlite3`, lo que la convierte en una gran opción para proyectos pequeños o medianos.

**Ventajas:**
- No necesita instalación adicional.
- Almacena los datos en un solo archivo `.db`.
- Ideal para pruebas, prototipos y aplicaciones de escritorio.


## 2. Conexión a la base de datos
Podemos conectarnos (o crear una nueva BD) usando `sqlite3.connect()`. Si el archivo no existe, se genera automáticamente.

In [2]:
import sqlite3

# Crear/abrir la base de datos
conexion = sqlite3.connect("ejemplo.db")

print("Conexión exitosa a la base de datos SQLite.")
conexion.close()

Conexión exitosa a la base de datos SQLite.


## 3. Creación de tablas
Definimos una tabla con columnas y tipos de datos.

In [3]:
conexion = sqlite3.connect("ejemplo.db")
cursor = conexion.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS estudiantes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    edad INTEGER,
    curso TEXT
)
''')

print("Tabla creada correctamente.")
conexion.commit()
conexion.close()

Tabla creada correctamente.


## 4. Operaciones CRUD
### 4.1 CREATE – Insertar registros

In [ ]:
conexion = sqlite3.connect("ejemplo.db")
cursor = conexion.cursor()

cursor.execute("INSERT INTO estudiantes (nombre, edad, curso) VALUES (?, ?, ?)", ("Ana", 20, "Python"))
cursor.execute("INSERT INTO estudiantes (nombre, edad, curso) VALUES (?, ?, ?)", ("Luis", 22, "Java"))
cursor.execute("INSERT INTO estudiantes (nombre, edad, curso) VALUES (?, ?, ?)", ("María", 21, "Python"))

conexion.commit()
print("Registros insertados correctamente.")
conexion.close()

También podemos ejecutar listas de datos con `executemany`:

In [4]:
usuarios_ejemplo = [
    ('Marta', 25, 'Python'),
    ('Carlos', 30, 'Java'),
    ('Sara', 22, 'Python'),
    ('Pedro', 35, 'Python')
]

conexion = sqlite3.connect("ejemplo.db")
cursor = conexion.cursor()

cursor.executemany('''
INSERT INTO estudiantes (nombre, edad, curso) 
VALUES (?, ?, ?)
''', usuarios_ejemplo)

conexion.commit()
print("Usuarios insertados")
conexion.close()

Usuarios insertados


### 4.2 READ – Consultar registros
Podemos recuperar datos usando `SELECT`. Existen varios métodos para obtener los resultados:

- `fetchone()` → devuelve una sola fila.
- `fetchmany(n)` → devuelve **n** filas.
- `fetchall()` → devuelve todas las filas disponibles.

In [5]:
conexion = sqlite3.connect("ejemplo.db")
cursor = conexion.cursor()

cursor.execute("SELECT * FROM estudiantes")

print("Un registro con fetchone():")
print(cursor.fetchone())

cursor.execute("SELECT * FROM estudiantes")
print("\nDos registros con fetchmany(2):")
print(cursor.fetchmany(2))

cursor.execute("SELECT * FROM estudiantes")
print("\nTodos los registros con fetchall():")
estudiantes = cursor.fetchall()
print("Lista de usuarios:")
for estudiante in estudiantes:
    print(f"ID: {estudiante[0]}, Nombre: {estudiante[1]}, Edad: {estudiante[2]}, Curso: {estudiante[3]}")

conexion.close()

Un registro con fetchone():
(1, 'Marta', 25, 'Python')

Dos registros con fetchmany(2):
[(1, 'Marta', 25, 'Python'), (2, 'Carlos', 30, 'Java')]

Todos los registros con fetchall():
Lista de usuarios:
ID: 1, Nombre: Marta, Edad: 25, Curso: Python
ID: 2, Nombre: Carlos, Edad: 30, Curso: Java
ID: 3, Nombre: Sara, Edad: 22, Curso: Python
ID: 4, Nombre: Pedro, Edad: 35, Curso: Python


### 4.3 UPDATE – Modificar registros

In [ ]:
conexion = sqlite3.connect("ejemplo.db")
cursor = conexion.cursor()

cursor.execute("UPDATE estudiantes SET curso = ? WHERE nombre = ?", ("Django", "Ana"))

conexion.commit()
print("Registro actualizado.")
conexion.close()

### 4.4 DELETE – Eliminar registros

In [ ]:
conexion = sqlite3.connect("ejemplo.db")
cursor = conexion.cursor()

cursor.execute("DELETE FROM estudiantes WHERE nombre = ?", ("Luis",))

conexion.commit()
print("Registro eliminado.")
conexion.close()

## 5. Ejemplos práctico: Gestor de Tareas

`gestor_tareas.py`

In [ ]:
import sqlite3
from datetime import datetime

class GestorTareas:
    def __init__(self, nombre_db='tareas.db'):
        self.conexion = sqlite3.connect(nombre_db)
        self.crear_tabla()
    
    def crear_tabla(self):
        cursor = self.conexion.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS tareas (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                descripcion TEXT NOT NULL,
                completada BOOLEAN DEFAULT FALSE,
                fecha_creacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        self.conexion.commit()
    
    def agregar_tarea(self, descripcion):
        cursor = self.conexion.cursor()
        cursor.execute('INSERT INTO tareas (descripcion) VALUES (?)', (descripcion,))
        self.conexion.commit()
        print(f"✅ Tarea agregada: {descripcion}")
    
    def listar_tareas(self, solo_pendientes=False):
        cursor = self.conexion.cursor()
        
        if solo_pendientes:
            cursor.execute('SELECT * FROM tareas WHERE completada = FALSE')
        else:
            cursor.execute('SELECT * FROM tareas')
        
        tareas = cursor.fetchall()
        
        if not tareas:
            print("📝 No hay tareas" + (" pendientes" if solo_pendientes else ""))
            return
        
        print(f"📋 Lista de tareas{' pendientes' if solo_pendientes else ''}:")
        print("-" * 50)
        for tarea in tareas:
            estado = "✅" if tarea[2] else "⏳"
            print(f"{tarea[0]}. {tarea[1]} {estado}")
    
    def completar_tarea(self, id_tarea):
        cursor = self.conexion.cursor()
        cursor.execute('UPDATE tareas SET completada = TRUE WHERE id = ?', (id_tarea,))
        
        if cursor.rowcount > 0:
            self.conexion.commit()
            print(f"✅ Tarea {id_tarea} marcada como completada")
        else:
            print("❌ Tarea no encontrada")
    
    def eliminar_tarea(self, id_tarea):
        cursor = self.conexion.cursor()
        cursor.execute('DELETE FROM tareas WHERE id = ?', (id_tarea,))
        
        if cursor.rowcount > 0:
            self.conexion.commit()
            print(f"✅ Tarea {id_tarea} eliminada")
        else:
            print("❌ Tarea no encontrada")
    
    def cerrar(self):
        self.conexion.close()

# Probemos nuestro gestor de tareas
if __name__ == "__main__":
    gestor = GestorTareas()
    
    # Agregar algunas tareas
    gestor.agregar_tarea("Aprender SQLite")
    gestor.agregar_tarea("Practicar Python")
    gestor.agregar_tarea("Hacer ejercicio")
    
    print("\\n" + "="*50)
    gestor.listar_tareas()
    
    # Completar una tarea
    print("\\n" + "="*50)
    gestor.completar_tarea(1)
    
    print("\\n" + "="*50)
    gestor.listar_tareas(solo_pendientes=True)
    
    gestor.cerrar()

## 6. Ejercicio propuesto
1. Crear una tabla llamada `cursos` con columnas `id`, `nombre`, `duracion`.
2. Insertar al menos **3 cursos**.
3. Mostrar todos los cursos con `fetchall()`.
4. Actualizar la duración de uno de ellos.
5. Eliminar un curso y comprobar que ya no aparece en los resultados.

## 📌 Resumen
- SQLite usa un solo archivo .db
- connect() para conectar
- cursor() para ejecutar comandos
- execute() para SQL
- commit() para guardar cambios
- fetchall() para obtener resultados